<a href="https://colab.research.google.com/github/Sitthisak17SM/Super_AI/blob/main/Word_Segmentation_600817.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, csv, re, time, requests
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import userdata

# Data

KAGGLE_API_KEY = userdata.get('KAGGLE_API')
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY
!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation
!unzip -q super-ai-engineer-ss-6-word-segmentation

super-ai-engineer-ss-6-word-segmentation.zip: Skipping, found more recently modified local copy (use --force to force download)
replace LST20 Annotation Guideline.pdf? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install pythainlp[full] -q

In [ ]:
!pip install deepcut -q

In [ ]:
!pip install pythainlp[full] attacut -q
!pip install deepcut -q

In [ ]:
import os, csv, re
import pandas as pd
from pathlib import Path
from pythainlp.tokenize import word_tokenize

In [ ]:
!pip install transformers datasets seqeval accelerate sentencepiece -q

In [ ]:
import tarfile
import os

os.makedirs("lst20_data", exist_ok=True)

# ใช้ tarfile.open พร้อมระบุโหมด "r:gz" สำหรับไฟล์ .tar.gz
with tarfile.open("/content/Corpus-LST20.tar.gz", "r:gz") as tar:
    tar.extractall(path="lst20_data/")

print("แตกไฟล์สำเร็จ!")

/tmp/ipykernel_15390/1926392832.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="lst20_data/")


แตกไฟล์สำเร็จ!


In [ ]:
# ============================================================
# Thai Word Segmentation — Super AI Engineer Competition
# Task: Tag each non-whitespace character with B_WORD / I_WORD / E_WORD
# Metric: F1-score (Macro)
# ============================================================

# ── 1. Install dependencies ━━━━━━━━━━━━━━━━━━━━
# !pip install pythainlp[full] attacut -q
# !pip install deepcut -q

import os
import glob
import pandas as pd
from collections import Counter
from pythainlp.tokenize import word_tokenize
from pythainlp.util import dict_trie
from pythainlp.corpus.common import thai_words

ENGINE = "ensemble"   # "attacut" | "deepcut" | "ensemble"
TEST_FILE   = "ws_test.txt"
SAMPLE_FILE = "ws_sample_submission.csv"
OUTPUT_FILE = f"submission_{ENGINE}_optimized.csv"

# ── 2. Build Custom Dictionary from LST20 ━━━━━━━━━━━━━━━━━━
print("Building Custom Dictionary from LST20...")
lst20_words = set()
train_files = glob.glob("lst20_data/**/train/*.txt", recursive=True)

if not train_files:
    train_files = glob.glob("lst20_data/**/*.txt", recursive=True)

for file_path in train_files:
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('\t')
                if len(parts) >= 1:
                    word = parts[0]
                    if word and not word.startswith('<') and word != "_" :
                        lst20_words.add(word)

print(f" → Extracted {len(lst20_words)} unique words from LST20.")

custom_words = set(thai_words())
custom_words.update(lst20_words)
custom_trie = dict_trie(dict_source=custom_words)
print(" → Custom Trie built successfully.\n")

# ── 3. Core tagging functions ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def tag_tokens(tokens: list) -> list:
    result = []
    for token in tokens:
        if token.strip() == "":
            for ch in token:
                result.append((ch, None))
        elif len(token) == 1:
            result.append((token, "B_WORD"))
        else:
            for i, ch in enumerate(token):
                if i == 0: tag = "B_WORD"
                elif i == len(token) - 1: tag = "E_WORD"
                else: tag = "I_WORD"
                result.append((ch, tag))
    return result

# แก้ไขฟังก์ชัน tokenise ให้รองรับ custom_dict
def tokenise(text: str, engine: str) -> list:
    print(f"  Tokenising with engine='{engine}' ...")

    # หากเป็น newmm ให้บังคับใช้ LST20 Dictionary ที่เราสร้างขึ้น
    if engine == "newmm":
        tokens = word_tokenize(text, engine=engine, custom_dict=lst20_trie, keep_whitespace=True)
    else:
        tokens = word_tokenize(text, engine=engine, keep_whitespace=True)

    tagged = tag_tokens(tokens) # เรียกใช้ฟังก์ชัน tag_tokens เดิมของคุณ

    assert len(tagged) == len(text), \
        f"[{engine}] Length mismatch: {len(tagged)} vs {len(text)}"
    print(f"  → {len(tokens)} tokens, {len(tagged)} chars tagged")
    return tagged
def post_process_tags(tagged_chars: list) -> list:
    reconstructed_tokens = []
    current_word = ""
    for ch, tag in tagged_chars:
        if tag is None:
            if current_word: reconstructed_tokens.append(current_word); current_word = ""
            reconstructed_tokens.append(ch)
        elif tag == "B_WORD":
            if current_word: reconstructed_tokens.append(current_word)
            current_word = ch
        else: current_word += ch
    if current_word: reconstructed_tokens.append(current_word)
    return tag_tokens(reconstructed_tokens)

# ── 4. Main Processing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
with open(TEST_FILE, "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total test characters: {len(raw_text)}\n")

if ENGINE == "ensemble":
    ENGINES = ["attacut", "deepcut", "newmm"]
    all_tagged = {}
    for eng in ENGINES: all_tagged[eng] = tokenise(raw_text, eng)
    print("  Voting...")
    raw_voted_tagged = []
    for i in range(len(raw_text)):
        ch = raw_text[i]
        votes = [all_tagged[eng][i][1] for eng in ENGINES]
        if all(v is None for v in votes): raw_voted_tagged.append((ch, None))
        else:
            non_null = [v for v in votes if v is not None]
            winner = Counter(non_null).most_common(1)[0][0]
            raw_voted_tagged.append((ch, winner))
    print("  Applying Post-processing to fix sequence logic...")
    tagged = post_process_tags(raw_voted_tagged)
    print(f"Ensemble done. Total positions: {len(tagged)}\n")
else:
    tagged = tokenise(raw_text, ENGINE)

# ── 5. Generate Submission File ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
sample = pd.read_csv(SAMPLE_FILE)
all_chars = []
for char_id, (ch, tag) in enumerate(tagged, start=1): all_chars.append({"Id": char_id, "char": ch, "Predicted": tag})
full_df = pd.DataFrame(all_chars)
sample_ids = set(sample["Id"])
df = full_df[full_df["Id"].isin(sample_ids)][["Id", "Predicted"]].copy()
df = sample[["Id"]].merge(df, on="Id", how="left")
df["Predicted"] = df["Predicted"].fillna("B_WORD")
df.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved optimized submission → {OUTPUT_FILE}")
print(df["Predicted"].value_counts())